In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
import time

from func_file_Aberrations import scaled_psf_vector

In [ ]:
#Create folder for storing generated PSFs
if not os.path.exists("Data_files"):
    os.mkdir("Data_files")
if not os.path.exists("Data_files/Aberrated_PSFs"):
    os.mkdir("Data_files/Aberrated_PSFs")

## Set the varibles

In [ ]:
#Set generating parameters
base_size = 50                    #Size of the matrix containing the aberrated PSF (default 50)
scale = 8                         #Upsampling of the targeted model (default 8)
up_size = base_size * scale

#Note:
#While the low resolution images lives on the (base_size X base_size) pixel size, the PSF convolution happens on the upsampled ground truth grid (up_size, up_size)
#Therefore, the PSFs must be also generated upscaled 

num_psfs = 21                     #Number of points on the horizontal axis (default 21)
max_aber = 0.2                    #Maximal aberration strength in its graph (default 0.2)
low_NA = 0.2                      #Minimal value of numerical aperture in its graph (default 0.2)
high_NA = 1.4                     #Maximal value of numerical aperture in its graph (default 1.4)

In [ ]:
#Set the dictionary for aberration types and strengths to generate
aberrations = {
    'A_coma_x' :  np.linspace(0.0, max_aber, num_psfs),
    'A_ast' : np.linspace(0.0, max_aber, num_psfs),
    'A_sph' : np.linspace(0.0, max_aber, num_psfs),    
    'A_def' : np.linspace(0.0, max_aber, num_psfs),
    'NA' : np.linspace(low_NA, high_NA, num_psfs)
}

In [ ]:
#Save after each type?
save_partial = False

#Print the progress?
print_progress = True

## Generate PSFs for aberrations

This generation process requires approximately an hour on our computational resources.  <br>

In [ ]:
#Allocate variables
psf_all_stacks = np.zeros((len(aberrations), num_psfs, up_size, up_size))
psf_current_stack = np.zeros((num_psfs, up_size, up_size))
A_list = [key for key, _ in aberrations.items()][:-1]

In [ ]:
#Generate the aberrated PSFs for the selected aberration types (without NA) and strengths

#Cycle over aberration types
for i, key in zip(range(len(A_list)), A_list):
    if print_progress:
        start = time.time()
        print("Preparing:", key)
    
    A_values = aberrations[key]
    
    #Cycle over aberration strengths
    for j in range(num_psfs):
        #Generate aberrated PSF for current aberration type and strength
        current_psf, _ = scaled_psf_vector(px_size_um=0.059/scale, 
                                           NA=1.4, 
                                           n=1.5, 
                                           N_out=50*scale,
                                           wavelength=0.532, 
                                           **{key : A_values[j]})
        psf_current_stack[j] = current_psf
        psf_all_stacks[i,j] = current_psf
        
        if print_progress:
            print("-> Finished", j+1, "out of", num_psfs, "in", np.round(time.time()-start, 1), "seconds.")
    
    if save_partial:
        np.savez("Data_files/Aberrated_PSFs/Stack_psf_for_" + key[2:] + "_aberrations_532nm", 
                 psf_stack = psf_current_stack, aberrations_strength = A_values)
    if print_progress:
        print("")

print("Process finished.")

## Generate PSFs for Numerical aperture

This generation process requires approximately 30 minutes on our computational resources.  <br>

In [ ]:
#Allocate variables
psf_stack = np.zeros([num_psfs, up_size, up_size])

In [ ]:
#Generate the aberrated PSFs for the selected values of the numerical aperture
NA_values = aberrations["NA"]

if print_progress:
    start = time.time()
    print("Preparing NA:")

#Cycle over NA values
for i in range(num_psfs):
    #Generate PSF for current NA value
    psf, _ = scaled_psf_vector(px_size_um=0.15/scale, 
                               NA=NA_values[i], 
                               n=1.5, 
                               wavelength=0.532, 
                               N_out=50*scale)
    psf_stack[i] = psf
    
    if print_progress:
        print("-> Finished", i+1, "out of", NA_values.shape[0], "in", np.round(time.time()-start, 1), "seconds.")

psf_all_stacks[-1] = psf_stack

if save_partial:
    np.savez("Data_files/Aberrated_PSFs/Stack_psf_for_NA_532nm.npz", 
             psf_stack = psf_stack, NA_values = NA_values)

## Save the generated PSFs

Stores approximately 150 MB.

In [ ]:
np.savez("Data_files/Aberrated_PSFs/All_stacks_psf_532nm.npz", 
         psf_all_stacks = psf_all_stacks, aberrations_dictionary = aberrations)

#Note: aberrations_dictionary is saved as object